# Olist E-Commerce Project Starter

Use this starter notebook for the BAN 6003 final project option: **Olist Brazilian E-Commerce**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset is relational. Your main job is to preserve the row meaning of your final ABT.

- A common final ABT is **one row per order**, identified by `order_id`.
- Aggregate `order_items`, `order_payments`, and `order_reviews` to `order_id` before joining them to `orders`.
- Join `products`, category translations, and sellers to item-level data before aggregating those fields to order level.
- `customer_id` identifies the customer record attached to one order; `customer_unique_id` may repeat because one person can place multiple orders.
- Check row counts and duplicate `order_id` values after every major merge.
- Define targets only where the outcome is observable. An undelivered order is not automatically an on-time order, and a missing review is not a low review.
- State when a prediction would be made, then exclude fields that would not be available at that time.


## Setup

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=[
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
customers = pd.read_csv(DATA_DIR / 'customers.csv')
items = pd.read_csv(DATA_DIR / 'order_items.csv', parse_dates=['shipping_limit_date'])
payments = pd.read_csv(DATA_DIR / 'order_payments.csv')
reviews = pd.read_csv(DATA_DIR / 'order_reviews.csv', parse_dates=['review_creation_date', 'review_answer_timestamp'])
products = pd.read_csv(DATA_DIR / 'products.csv')
sellers = pd.read_csv(DATA_DIR / 'sellers.csv')
geo = pd.read_csv(DATA_DIR / 'geolocation_zip_prefix.csv')
translations = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

tables = {
    'orders': orders,
    'customers': customers,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
    'geolocation_zip_prefix': geo,
    'translations': translations,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


orders: 25,000 rows x 8 columns


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,3b697a20d9e427646d92567910af6d57,355077684019f7f60a031656bd7262b8,delivered,2016-10-03 09:44:50,2016-10-06 15:50:54,2016-10-23 14:02:13,2016-10-26 14:02:13,2016-10-27
1,cd3b8574c82b42fc8129f6d502690c3e,7812fcebfc5e8065d31e1bb5f0017dae,delivered,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23
2,1aecadf4362edaca7fa033e882076c8d,e81a9f176936e3124dfd90c483bf3289,canceled,2016-10-04 10:05:45,2016-10-04 10:26:40,NaT,NaT,2016-11-24



customers: 25,000 rows x 5 columns


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
2,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



items: 28,343 rows x 7 columns


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP



geolocation_zip_prefix: 10,009 rows x 6 columns


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,geolocation_records
0,1001,-23.550190,-46.634024,sao paulo,SP,26
1,1004,-23.549799,-46.634757,sao paulo,SP,22
2,1005,-23.549456,-46.636733,sao paulo,SP,25



translations: 71 rows x 2 columns


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,orders,25000,8,0,1266
1,customers,25000,5,0,0
2,items,28343,7,0,0
3,payments,26154,5,0,0
4,reviews,24934,7,0,36618
5,products,13344,9,0,924
6,sellers,2234,4,0,0
7,geolocation_zip_prefix,10009,6,0,0
8,translations,71,2,0,0


### Key and Granularity Audit

Confirm the expected key for each table before planning joins. A duplicate `order_id` is expected in item, payment, and review detail tables because those tables can contain several records for one order. Their more detailed keys should still be checked. The provided geolocation table is already summarized to one row per zip-code prefix.


In [5]:
key_audit = pd.DataFrame([
    {"table": "orders", "expected_key": "order_id", "duplicate_keys": orders["order_id"].duplicated().sum(), "missing_key_values": orders["order_id"].isna().sum()},
    {"table": "customers", "expected_key": "customer_id", "duplicate_keys": customers["customer_id"].duplicated().sum(), "missing_key_values": customers["customer_id"].isna().sum()},
    {"table": "order_items", "expected_key": "order_id + order_item_id", "duplicate_keys": items.duplicated(["order_id", "order_item_id"]).sum(), "missing_key_values": items[["order_id", "order_item_id"]].isna().any(axis=1).sum()},
    {"table": "order_payments", "expected_key": "order_id + payment_sequential", "duplicate_keys": payments.duplicated(["order_id", "payment_sequential"]).sum(), "missing_key_values": payments[["order_id", "payment_sequential"]].isna().any(axis=1).sum()},
    {"table": "order_reviews", "expected_key": "review record (review_id may repeat)", "duplicate_keys": reviews.duplicated().sum(), "missing_key_values": reviews["order_id"].isna().sum()},
    {"table": "products", "expected_key": "product_id", "duplicate_keys": products["product_id"].duplicated().sum(), "missing_key_values": products["product_id"].isna().sum()},
    {"table": "sellers", "expected_key": "seller_id", "duplicate_keys": sellers["seller_id"].duplicated().sum(), "missing_key_values": sellers["seller_id"].isna().sum()},
    {"table": "geolocation_zip_prefix", "expected_key": "geolocation_zip_code_prefix", "duplicate_keys": geo["geolocation_zip_code_prefix"].duplicated().sum(), "missing_key_values": geo["geolocation_zip_code_prefix"].isna().sum()},
    {"table": "translations", "expected_key": "product_category_name", "duplicate_keys": translations["product_category_name"].duplicated().sum(), "missing_key_values": translations["product_category_name"].isna().sum()},
])

display(key_audit)
print(f"Item rows sharing an order_id: {items['order_id'].duplicated().sum():,}")
print(f"Payment rows sharing an order_id: {payments['order_id'].duplicated().sum():,}")
print(f"Review rows sharing an order_id: {reviews['order_id'].duplicated().sum():,}")
print(f"Repeated customer_unique_id values: {customers['customer_unique_id'].duplicated().sum():,}")


,table,expected_key,duplicate_keys,missing_key_values
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,order_items,order_id + order_item_id,0,0
3,order_payments,order_id + payment_sequential,0,0
4,order_reviews,review record (review_id may repeat),0,0
5,products,product_id,0,0
6,sellers,seller_id,0,0
7,geolocation_zip_prefix,geolocation_zip_code_prefix,0,0
8,translations,product_category_name,0,0


Item rows sharing an order_id: 3,552
Payment rows sharing an order_id: 1,154
Review rows sharing an order_id: 137
Repeated customer_unique_id values: 236


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Olist notes:

- Keep a table inventory: what does one row mean in each file?
- `order_items`, `order_payments`, and `order_reviews` are not guaranteed to be one row per order.
- Missing review comments are common and should be described, not automatically deleted.
- Product categories may need translation using `product_category_name_translation.csv`.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

#### Business or Research Question

The primary research question for this project is:

**How does delivery performance relate to customer review scores in the Olist e-commerce marketplace?**

I am interested in determining whether customers who experience longer delivery times or late deliveries are more likely to provide lower review scores. I will also examine whether other order characteristics, such as freight cost, product category, seller information, and order value, are associated with customer satisfaction.

#### Why This Question Matters

Delivery performance is an important part of the customer experience in e-commerce. Customers generally expect their orders to arrive within the estimated delivery period, and delays may negatively affect their satisfaction with the marketplace.

An operations or customer experience team could use this analysis to identify patterns associated with poor reviews and determine whether fulfillment and delivery performance are areas where improvements could increase customer satisfaction.

The intended users of the analysis are marketplace operations managers, logistics teams, and customer experience analysts.

#### Dataset Source

This project uses the Olist Brazilian E-Commerce dataset provided through the BAN 6003 Option 2 project template:

https://github.com/tianhaiz/ban6003-project-option-2-olist-template

The project repository contains the curated CSV files used for this analysis.

#### Tables and Files Used

The project contains nine related data files:

- `orders.csv`
- `customers.csv`
- `order_items.csv`
- `order_payments.csv`
- `order_reviews.csv`
- `products.csv`
- `sellers.csv`
- `geolocation_zip_prefix.csv`
- `product_category_name_translation.csv`

These files contain information about orders, customers, products, sellers, payments, reviews, delivery dates, and geographic information.

#### Row Meaning of Each Important Table

- **orders.csv:** One row represents one order. The primary identifier is `order_id`.
- **customers.csv:** One row represents one order-specific customer record. The expected key is `customer_id`. A `customer_unique_id` may appear multiple times when the same customer places multiple orders.
- **order_items.csv:** One row represents one numbered item line within an order. The expected key is the combination of `order_id` and `order_item_id`.
- **order_payments.csv:** One row represents one payment transaction or payment-method sequence associated with an order. An order may therefore contain multiple payment rows.
- **order_reviews.csv:** One row represents one review record associated with an order. Review records are not guaranteed to be unique by `order_id` or `review_id` alone.
- **products.csv:** One row represents one product. The expected key is `product_id`.
- **sellers.csv:** One row represents one seller. The expected key is `seller_id`.
- **geolocation_zip_prefix.csv:** One row represents one summarized geographic ZIP-code prefix.
- **product_category_name_translation.csv:** One row represents one mapping between a Portuguese product category name and its English translation.

Because several tables can contain multiple records for the same order, they cannot all be directly merged with the orders table while still assuming that one row represents one order. Item, payment, and review information will need to be aggregated to the order level before the final analytic base table is constructed.

#### Early Data Quality Issues

Initial profiling identified several important data-quality and structural issues.

There are no exact duplicate rows in any of the nine source tables. The expected keys checked in the starter notebook also do not contain unexpected duplicate or missing key values. However, repeated `order_id` values are expected in detail-level tables such as order items, payments, and reviews because these tables may contain multiple records for a single order.

Missing values are concentrated in several important areas. The orders table contains 775 missing customer delivery dates, 452 missing carrier delivery dates, and 39 missing order approval dates. These values should not automatically be treated as errors because some orders may have been canceled, unavailable, or otherwise not completed.

Missing review text is especially common. There are 22,008 missing review titles, representing 88.27% of review records, and 14,610 missing review messages, representing 58.59%. However, the review score itself is present and all observed scores fall within the expected range of 1 through 5. Therefore, missing written comments should not be interpreted as missing or negative reviews.

The products table contains 230 missing product category names and related descriptive fields. There is also one missing value in each of the product weight and dimension fields. These missing product attributes may affect analyses involving product characteristics or product categories.

Delivery-date validation found no orders where the customer delivery date occurred before the purchase date. Among orders with a recorded customer delivery date, the average delivery time is approximately 12 days, while the longest observed delivery time is 208 days. A total of 1,593 orders were delivered after their estimated delivery date. These late deliveries are especially relevant to the project's research question concerning delivery performance and customer satisfaction.

The review table also requires special handling before it can be joined to an order-level analytic table. There are 136 orders with more than one review record, including one order with three review records. Review information will therefore need to be aggregated to `order_id` before being merged into a one-row-per-order analytic base table.

Some extreme numeric values, such as unusually large product measurements, prices, freight costs, or long delivery periods, may require additional investigation. These values will not automatically be removed because an extreme value may represent a legitimate transaction rather than a data-entry error.

#### Initial Cleaning Plan

My initial cleaning plan is to:

1. Verify the expected key and row meaning of every table.
2. Check for exact duplicate records and unexpected duplicate keys.
3. Examine missing values and determine whether they represent errors or valid business situations.
4. Validate that review scores fall within the expected range.
5. Check prices, freight values, product measurements, and other numeric variables for impossible or suspicious values.
6. Check the chronological order of purchase, approval, shipment, estimated delivery, and actual delivery dates.
7. Create useful delivery measures such as total delivery time and whether an order was delivered after its estimated delivery date.
8. Translate product category names where needed using the provided translation table.
9. Aggregate item, payment, and review information to `order_id` before joining those tables to the order-level data.
10. Validate row counts and duplicate `order_id` values after future joins to ensure that the final analytic base table remains one row per order.

Unusual or extreme values will be investigated before they are removed because an unusual value may represent a legitimate transaction rather than an error.

#### Possible Target or Outcome

The primary outcome I plan to study is **customer review score**.

The original `review_score` variable can be used as an outcome for analyzing customer satisfaction. An alternative later in the project may be to create a categorical variable such as `high_review_score`.

Potential explanatory variables include:

- delivery time
- whether an order was delivered late
- freight cost
- total order value
- product category
- number of products in an order
- seller information
- customer location
- seller location

A review-based outcome will only be defined for orders where a review score is actually observed. A missing review will not automatically be interpreted as a low review.

#### Ethics, Privacy, and Governance Concerns

Although the Olist dataset is publicly available, the analysis should still be conducted responsibly.

Customer and seller identifiers will be treated as analytical identifiers rather than used to attempt to identify individuals or businesses. Results should generally focus on patterns and groups rather than individual customers.

Geographic differences should also be interpreted carefully. Longer delivery times in certain areas may result from transportation distance, infrastructure, seller availability, or logistics networks and should not automatically be attributed to characteristics of customers living in those areas.

Low review scores also should not automatically be blamed on individual sellers because customer satisfaction may be affected by several factors outside the seller's control, including carriers, delivery distance, damaged packages, marketplace operations, or customer expectations.

Missing data will be examined in its business context rather than automatically deleted. Cleaning decisions, transformations, and future joins will be documented so that the project remains reproducible.

In [15]:
# Milestone 1 code workspace
# Add your additional profiling checks here.
# Examples: missingness by important columns, duplicate key checks, suspicious value checks.

In [7]:
# Milestone 1 additional profiling checks

# 1. Missing values by table
for name, df in tables.items():
    print(f"\n{name.upper()} - MISSING VALUES")
    
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2)
    })
    
    display(
        missing[missing["missing_count"] > 0]
        .sort_values("missing_count", ascending=False)
    )


ORDERS - MISSING VALUES


,missing_count,missing_percent
order_delivered_customer_date,775,3.10
order_delivered_carrier_date,452,1.81
order_approved_at,39,0.16



CUSTOMERS - MISSING VALUES


,missing_count,missing_percent



ITEMS - MISSING VALUES


...

,missing_count,missing_percent



GEOLOCATION_ZIP_PREFIX - MISSING VALUES


,missing_count,missing_percent



TRANSLATIONS - MISSING VALUES


,missing_count,missing_percent


In [8]:
# 2. Order status and review score distributions

print("ORDER STATUS")
display(orders["order_status"].value_counts(dropna=False).to_frame("count"))

print("\nREVIEW SCORES")
display(
    reviews["review_score"]
    .value_counts(dropna=False)
    .sort_index()
    .to_frame("count")
)

ORDER STATUS


,count
order_status,
delivered,24222
shipped,309
unavailable,167
canceled,154
invoiced,75
processing,71
approved,1
created,1



REVIEW SCORES


,count
review_score,
1,2831
2,793
3,2019
4,4780
5,14511


In [9]:
# 3. Check review score validity

invalid_reviews = reviews[
    ~reviews["review_score"].between(1, 5)
]

print("Review scores outside 1-5:", len(invalid_reviews))
display(invalid_reviews.head())

Review scores outside 1-5: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [10]:
# 4. Check important numeric fields

print("ORDER ITEM PRICE AND FREIGHT")
display(
    items[["price", "freight_value"]].describe()
)

print("\nPRODUCT MEASUREMENTS")
display(
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ].describe()
)

ORDER ITEM PRICE AND FREIGHT


,price,freight_value
count,28343.000000,28343.000000
mean,120.529931,19.973035
std,178.492390,15.387608
min,2.200000,0.000000
25%,39.900000,13.125000
50%,75.000000,16.320000
75%,135.000000,21.235000
max,6735.000000,339.590000



PRODUCT MEASUREMENTS


,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,13343.000000,13343.000000,13343.000000,13343.000000
mean,2192.937570,30.574758,16.788953,23.152140
std,4052.917466,16.616449,13.525831,12.097029
min,2.000000,7.000000,2.000000,7.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1850.000000,38.000000,20.000000,30.000000
max,30000.000000,105.000000,105.000000,118.000000


In [11]:
# 5. Initial delivery-date validation

delivered_orders = orders[
    orders["order_delivered_customer_date"].notna()
].copy()

delivered_orders["delivery_days"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_purchase_timestamp"]
).dt.days

delivered_orders["days_vs_estimate"] = (
    delivered_orders["order_delivered_customer_date"]
    - delivered_orders["order_estimated_delivery_date"]
).dt.days

print("DELIVERY DAYS")
display(delivered_orders["delivery_days"].describe())

print(
    "Orders with delivery before purchase:",
    (delivered_orders["delivery_days"] < 0).sum()
)

print(
    "Orders delivered after estimated date:",
    (delivered_orders["days_vs_estimate"] > 0).sum()
)

DELIVERY DAYS


count    24225.000000
mean        12.007224
std          9.300737
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        208.000000
Name: delivery_days, dtype: float64

Orders with delivery before purchase: 0
Orders delivered after estimated date: 1593


In [12]:
# 6. Review records per order

review_counts = (
    reviews.groupby("order_id")
    .size()
    .sort_values(ascending=False)
)

print("Orders with more than one review record:",
      (review_counts > 1).sum())

display(review_counts.head(10))

Orders with more than one review record: 136


order_id
8e17072ec97ce29f0e1f111e598b0c85    3
3fe4dbcdb046a475dbf25463c1ca78bd    2
9d6e7e70e068602fa740fc3163cefd3b    2
3b132068223649635448d659e7826d12    2
62adfd8a504d828f9dc9c184f2a236b8    2
5f92ad7a8f00f12846cf6f654966adc4    2
142b0e3de54c110a3109b26c79d6ec2e    2
361e0cf744333677864a5e28558a53bc    2
1aa720b6963ff9b4b5bff9bc48b49a89    2
78cd965d0bc0388d390404eee6490c5b    2
dtype: int64

In [13]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


orders


,dtype
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,datetime64[us]
order_approved_at,datetime64[us]
order_delivered_carrier_date,datetime64[us]
order_delivered_customer_date,datetime64[us]
order_estimated_delivery_date,datetime64[us]


,missing_rate
order_delivered_customer_date,0.03100
order_delivered_carrier_date,0.01808
order_approved_at,0.00156
order_id,0.00000
order_purchase_timestamp,0.00000
order_status,0.00000
customer_id,0.00000
order_estimated_delivery_date,0.00000



customers


,dtype
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str


...

,dtype
geolocation_zip_code_prefix,int64
geolocation_lat,float64
geolocation_lng,float64
geolocation_city,str
geolocation_state,str
geolocation_records,int64


,missing_rate
geolocation_zip_code_prefix,0.0
geolocation_lat,0.0
geolocation_lng,0.0
geolocation_city,0.0
geolocation_state,0.0
geolocation_records,0.0



translations


,dtype
product_category_name,str
product_category_name_english,str


,missing_rate
product_category_name,0.0
product_category_name_english,0.0


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what key defines one row, and which tables were joined or aggregated.

Critical integration reminders:

- Enrich item records with product, category, and seller attributes while they are still item-level.
- Aggregate item, payment, and review records separately to `order_id`.
- Join each order-level summary to `orders` with merge validation.
- Treat the order-level ABT as trustworthy only after its row count and unique key have been checked.

The starter below demonstrates this sequence. Revise the variables to fit your question, and explain why each retained variable is useful.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [ ]:
# Milestone 2 code workspace
# Build or revise your ABT here.
# Add row-count checks and duplicate-key checks after each major merge.



In [ ]:
products_enriched = products.merge(
    translations,
    on="product_category_name",
    how="left",
    validate="many_to_one",
)

item_detail = (
    items
    .merge(
        products_enriched[[
            "product_id", "product_category_name_english", "product_weight_g"
        ]],
        on="product_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]],
        on="seller_id",
        how="left",
        validate="many_to_one",
    )
)

item_summary = (
    item_detail.groupby("order_id")
    .agg(
        order_item_count=("order_item_id", "count"),
        product_count=("product_id", "nunique"),
        seller_count=("seller_id", "nunique"),
        category_count=("product_category_name_english", "nunique"),
        seller_state_count=("seller_state", "nunique"),
        item_price_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
        product_weight_mean_g=("product_weight_g", "mean"),
    )
    .reset_index()
)

payment_summary = (
    payments.groupby("order_id")
    .agg(
        payment_record_count=("payment_sequential", "count"),
        payment_total=("payment_value", "sum"),
        payment_installments_max=("payment_installments", "max"),
        payment_type_count=("payment_type", "nunique"),
    )
    .reset_index()
)

reviews_for_summary = reviews.assign(
    review_comment_present=reviews["review_comment_message"].notna().astype(int)
)
review_summary = (
    reviews_for_summary.groupby("order_id")
    .agg(
        review_record_count=("review_id", "size"),
        review_score_mean=("review_score", "mean"),
        review_comment_present=("review_comment_present", "max"),
    )
    .reset_index()
)

customer_geo = customers.merge(
    geo[[
        "geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng",
        "geolocation_records"
    ]],
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left",
    validate="many_to_one",
).rename(columns={
    "geolocation_lat": "customer_lat",
    "geolocation_lng": "customer_lng",
    "geolocation_records": "customer_geo_source_records",
})

abt = (
    orders
    .merge(customer_geo, on="customer_id", how="left", validate="one_to_one")
    .merge(item_summary, on="order_id", how="left", validate="one_to_one")
    .merge(payment_summary, on="order_id", how="left", validate="one_to_one")
    .merge(review_summary, on="order_id", how="left", validate="one_to_one")
)

abt["purchase_month"] = abt["order_purchase_timestamp"].dt.month
abt["purchase_dayofweek"] = abt["order_purchase_timestamp"].dt.dayofweek
abt["promised_delivery_days"] = (
    abt["order_estimated_delivery_date"] - abt["order_purchase_timestamp"]
).dt.days
abt["delivery_days"] = (
    abt["order_delivered_customer_date"] - abt["order_purchase_timestamp"]
).dt.days

delivery_outcome_observed = (
    abt["order_status"].eq("delivered")
    & abt["order_delivered_customer_date"].notna()
    & abt["order_estimated_delivery_date"].notna()
)
abt["delivered_late"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
abt.loc[delivery_outcome_observed, "delivered_late"] = (
    abt.loc[delivery_outcome_observed, "order_delivered_customer_date"]
    > abt.loc[delivery_outcome_observed, "order_estimated_delivery_date"]
).astype(int)

abt["high_review_score"] = pd.Series(pd.NA, index=abt.index, dtype="Int64")
review_outcome_observed = abt["review_score_mean"].notna()
abt.loc[review_outcome_observed, "high_review_score"] = (
    abt.loc[review_outcome_observed, "review_score_mean"] >= 4
).astype(int)


In [ ]:
abt_validation = pd.Series({
    "orders_before_merges": len(orders),
    "abt_rows_after_merges": len(abt),
    "unique_order_ids": abt["order_id"].nunique(),
    "duplicate_order_ids": abt["order_id"].duplicated().sum(),
    "orders_with_item_summary": abt["order_item_count"].notna().sum(),
    "orders_with_payment_summary": abt["payment_total"].notna().sum(),
    "orders_with_review_summary": abt["review_score_mean"].notna().sum(),
    "orders_with_observed_delivery_target": abt["delivered_late"].notna().sum(),
    "orders_with_observed_review_target": abt["high_review_score"].notna().sum(),
})

display(abt_validation.to_frame("value"))
display(abt.head())
display(abt.isna().mean().sort_values(ascending=False).head(15).to_frame("missing_rate"))


### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the order-level ABT row count, key uniqueness, and target eligibility without creating another permanent database file.


In [ ]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("olist_order_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        '''
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT order_id) AS unique_order_ids,
            SUM(CASE WHEN delivered_late IS NOT NULL THEN 1 ELSE 0 END) AS delivery_target_rows,
            SUM(CASE WHEN high_review_score IS NOT NULL THEN 1 ELSE 0 END) AS review_target_rows
        FROM olist_order_abt
        ''',
        connection,
    )

display(sql_abt_check)


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [ ]:
data_dictionary = pd.DataFrame({
    "column_name": abt.columns,
    "data_type": [str(abt[column].dtype) for column in abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary.head()


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminders:

- If you model `delivered_late`, use only rows where the nullable target is defined; undelivered orders do not reveal a normal late/on-time outcome.
- For a prediction made at purchase time, do not use actual delivery duration, final order status, review fields, or other post-purchase outcomes as predictors.
- If you model `high_review_score`, use only rows with an observed review and do not use review-derived fields as predictors.
- Check class balance, keep a reproducible train/test split, and interpret precision and recall when the positive class is uncommon.
- Explain the intended decision and prediction timing before choosing features.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [ ]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


In [ ]:
# Example placeholder. Replace with your chosen target and feature set.
# target = 'your_target_column'
# features = ['feature_1', 'feature_2']
# model_df = abt[[target] + features].dropna()

abt.describe(include='all').T.head(25)

## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [ ]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "olist_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "olist_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "olist_model_metrics.csv"
# report_path = REPORTS_DIR / "olist_final_report.html"


In [ ]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "olist_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "olist_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
